In [ ]:
! pip install ultralytics
! pip install pafy
! pip install youtube-dl
! pip install yt-dlp


In [ ]:
import subprocess
import cv2
from ultralytics import YOLO
from google.colab.patches import cv2_imshqow

# 1. Configuration
YOUTUBE_URL = "https://www.youtube.com/live/BN7gzH-i-zo?feature=shared"
MODEL_PATH  = "/content/best.pt"   # 挂载在 block volume 上的模型路径

# 2. Use yt-dlp to get the direct video URL
#    -f best  选最高质量
#    -g       只输出可播放的 URL(s)
proc = subprocess.run(
    ["yt-dlp", "-f", "best", "-g", YOUTUBE_URL],
    capture_output=True,
    text=True,
    check=True
)
stream_urls = proc.stdout.splitlines()
# 如果列表里有多个 URL，通常第一个就是视频流
stream_url = stream_urls[0]

# 3. 打开流
cap = cv2.VideoCapture(stream_url)
if not cap.isOpened():
    raise RuntimeError("无法打开视频流")

# 4. 加载 YOLO 模型
model = YOLO(MODEL_PATH)

# 5. 逐帧检测并画框
while True:
    ret, frame = cap.read()
    if not ret: break

    results = model(frame)[0]  # ultralytics 执行推理
    for box in results.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        conf = float(box.conf[0])
        cls  = int(box.cls[0])
        label = f"{model.names[cls]} {conf:.2f}"
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)
        cv2.putText(frame, label, (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1)

    cv2_imshow(frame)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()


ImportError: cannot import name 'cv2_imshqow' from 'google.colab.patches' (/usr/local/lib/python3.11/dist-packages/google/colab/patches/__init__.py)